In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))
from paths import *

In [ ]:
# Create directory structure
for split in ["Train", "Test"]:
    os.makedirs(os.path.join(YOLO_RESEARCH_DATASET, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(YOLO_RESEARCH_DATASET, split, "labels"), exist_ok=True)

In [ ]:
# Helper function to generate YOLO annotation from a mask
def generate_yolo_annotations(mask: np.ndarray) -> list[str]:
    """
    Generate YOLO-format annotations from a binary segmentation mask.

    This function takes a binary mask and extracts external contours to generate YOLO-style 
    polygon annotations. The annotations are normalized to the image dimensions and assume 
    a single class (class ID 0).

    Args:
        mask (np.ndarray): A 2D numpy array representing the binary mask of the object(s). 
                           Pixel values should be 0 or 1.

    Returns:
        List[str]: A list of YOLO polygon annotation strings. Each string starts with the 
                   class ID '0' followed by normalized (x, y) coordinates of the polygon points.
                   Returns an empty list if no valid contours are found.
    
    Note:
        - Only external contours are considered (using cv2.RETR_EXTERNAL).
        - Contours with fewer than 3 points are ignored (not valid polygons).
        - Coordinates are normalized to the range [0, 1] relative to the image width and height.
    """
    height, width = mask.shape

    # Ensure binary mask
    mask = mask * 255
    mask = mask.astype(np.uint8)
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []

    yolo_annotations = []
    for contour in contours:
        if len(contour) < 3:
            continue  # Skip invalid polygons

        polygon_normalized = [(x / width, y / height) for [[x, y]] in contour]
        annotation_line = "0 " + " ".join([f"{x:.6f} {y:.6f}" for x, y in polygon_normalized])
        yolo_annotations.append(annotation_line)

    return yolo_annotations

In [ ]:
# Process both splits
for split in ["Train", "Test"]:
    src_dir = Path(MASKRCNN_RESEARCH_DATASET) / split
    dst_images = Path(YOLO_RESEARCH_DATASET) / split.lower() / "images"
    dst_labels = Path(YOLO_RESEARCH_DATASET) / split.lower() / "labels"

    # Collect all mask-image pairs
    for item in tqdm(os.listdir(src_dir), desc=f"Processing {split}"):
        item_path = src_dir / item
        if item.endswith("mask.png"):
            image_path = str(item_path).replace("mask.png", "image.png")
            image_name = os.path.basename(image_path)
            label_name = image_name.replace(".png", ".txt")

            # Load image and mask
            image = cv2.imread(image_path)
            mask = cv2.imread(str(item_path), cv2.IMREAD_GRAYSCALE)

            if image is None or mask is None:
                print(f"Skipping {image_path}, file not found or corrupted.")
                continue

            # Generate annotations and write them to the dataset
            annotations = generate_yolo_annotations(mask)
            cv2.imwrite(str(dst_images / image_name), image)
            with open(dst_labels / label_name, "w") as f:
                f.write("\n".join(annotations))
